In [2]:
import pandas as pd

df = pd.read_csv('archive/merged_output_ready.csv')
print(df)

                timestamp  ADALAR_BOSTANCI  ADALAR_BURGAZADA  ADALAR_BUYUKADA  \
0     2023-01-01 00:00:00         0.000000          0.000000         0.000000   
1     2023-01-01 01:00:00         0.000000          0.000000         0.000000   
2     2023-01-01 02:00:00         0.000000          0.000000         0.000000   
3     2023-01-01 03:00:00         0.000000          0.000000         0.000000   
4     2023-01-01 04:00:00         0.000000          0.000000         0.000000   
...                   ...              ...               ...              ...   
8734  2023-12-31 19:00:00         0.090103          0.032389         0.064713   
8735  2023-12-31 20:00:00         0.036928          0.000000         0.038665   
8736  2023-12-31 21:00:00         0.028065          0.020243         0.031339   
8737  2023-12-31 22:00:00         0.026588          0.000000         0.001221   
8738  2023-12-31 23:00:00         0.000492          0.000000         0.000000   

      ADALAR_HEYBELIADA  AD

In [4]:
print("--- Sütun İsimleri ve Veri Tipleri ---")
print(df.info())
print("\n--- İlk 5 Satır Örneği ---")
print(df)

--- Sütun İsimleri ve Veri Tipleri ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8739 entries, 0 to 8738
Columns: 484 entries, timestamp to hour
dtypes: float64(480), int64(3), object(1)
memory usage: 32.3+ MB
None

--- İlk 5 Satır Örneği ---
                timestamp  ADALAR_BOSTANCI  ADALAR_BURGAZADA  ADALAR_BUYUKADA  \
0     2023-01-01 00:00:00         0.000000          0.000000         0.000000   
1     2023-01-01 01:00:00         0.000000          0.000000         0.000000   
2     2023-01-01 02:00:00         0.000000          0.000000         0.000000   
3     2023-01-01 03:00:00         0.000000          0.000000         0.000000   
4     2023-01-01 04:00:00         0.000000          0.000000         0.000000   
...                   ...              ...               ...              ...   
8734  2023-12-31 19:00:00         0.090103          0.032389         0.064713   
8735  2023-12-31 20:00:00         0.036928          0.000000         0.038665   
8736  2023-12-31 21:

In [6]:
pip install pandas numpy scikit-learn xgboost tensorflow

  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\Mutlu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

print(" Veri setini yüklüyorum...")
df = pd.read_csv('archive/merged_output_ready.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

df['month'] = df['timestamp'].dt.month
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['hour'] = df['timestamp'].dt.hour
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

features = ['month', 'day_of_week', 'hour', 'is_weekend']
target_station = 'YENIKAPI - HAVALIMANI_ZEYTINBURNU' # Hedef Durağımız

X = df[features]
y = df[target_station]

# Eğitim/Test Ayrımı (%80 Eğitim, %20 Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = {} 


In [ ]:

print("\n Random Forest Eğitiliyor...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
results['Random Forest'] = mean_absolute_error(y_test, rf_preds)


In [ ]:

print(" XGBoost Eğitiliyor...")
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
results['XGBoost'] = mean_absolute_error(y_test, xgb_preds)


In [ ]:

print("📐 SVM (SVR) Eğitiliyor... (Bu biraz sürebilir)")
svm_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svm_model.fit(X_train_scaled, y_train)
svm_preds = svm_model.predict(X_test_scaled)
results['SVM'] = mean_absolute_error(y_test, svm_preds)


In [ ]:

print(" DNN (3 Katmanlı) Eğitiliyor...")

dnn_model = Sequential()
# 1. Katman (Giriş + Gizli Katman)
dnn_model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))
# 2. Katman
dnn_model.add(Dense(32, activation='relu'))
# 3. Katman
dnn_model.add(Dense(16, activation='relu'))
# Çıkış Katmanı (Tek bir sayı tahmin ediyoruz)
dnn_model.add(Dense(1, activation='linear')) 

dnn_model.compile(loss='mean_squared_error', optimizer=Adam(learning_rate=0.001))

dnn_model.fit(X_train_scaled, y_train, epochs=50, batch_size=32, verbose=0)

dnn_preds = dnn_model.predict(X_test_scaled).flatten()
results['DNN (3-Layer)'] = mean_absolute_error(y_test, dnn_preds)


In [ ]:

print("\n" + "="*40)
print(" MODELLERİN HATA SKORLARI (Daha düşük = Daha iyi)")
print("="*40)

sorted_results = dict(sorted(results.items(), key=lambda item: item[1]))

for name, score in sorted_results.items():
    print(f"{name:<20} : {score:.5f}")

winner = list(sorted_results.keys())[0]
print(f"\n🎉 KAZANAN MODEL: {winner}")
print("="*40)

if winner == 'XGBoost':
    joblib.dump(xgb_model, 'best_model.pkl')
elif winner == 'Random Forest':
    joblib.dump(rf_model, 'best_model.pkl')

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb  
import joblib
from tqdm import tqdm  

print(" Veri yükleniyor...")
df = pd.read_csv('archive/merged_output_ready.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['month'] = df['timestamp'].dt.month
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['hour'] = df['timestamp'].dt.hour
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

features = ['month', 'day_of_week', 'hour', 'is_weekend']
X = df[features]

# Hedef Sütunları Belirle (timestamp ve feature'lar hariç hepsi duraktır)
non_station_cols = features + ['timestamp']
station_cols = [col for col in df.columns if col not in non_station_cols]

print(f" Toplam {len(station_cols)} durak için yapay zeka ajanları eğitilecek...")

all_models = {}

# tqdm ile ilerleme çubuğu görelim
for station in tqdm(station_cols):
    y = df[station]
    
    model = xgb.XGBRegressor(n_estimators=50, learning_rate=0.1, n_jobs=-1, random_state=42)
    model.fit(X, y)

    all_models[station] = model

print(" Tüm modeller 'all_istanbul_models.pkl' dosyasına kaydediliyor...")
joblib.dump(all_models, 'all_istanbul_models.pkl')
print(" İŞLEM TAMAM! Tüm şehrin beyni artık dosyanızda.")

 Veri yükleniyor...
 Toplam 480 durak için yapay zeka ajanları eğitilecek...


100%|██████████| 480/480 [00:21<00:00, 22.76it/s]


 Tüm modeller 'all_istanbul_models.pkl' dosyasına kaydediliyor...
 İŞLEM TAMAM! Tüm şehrin beyni artık dosyanızda.


In [11]:
df

,timestamp,ADALAR_BOSTANCI,ADALAR_BURGAZADA,ADALAR_BUYUKADA,ADALAR_HEYBELIADA,ADALAR_KARTAL,ADALAR_KINALIADA,ADALAR-BOSTANCI_BURGAZADA,ADALAR-BOSTANCI_BUYUKADA,ADALAR-BOSTANCI_HEYBELIADA,...,YENIKAPI - HAVALIMANI_ULUBATLI,YENIKAPI - HAVALIMANI_YENIBOSNA,YENIKAPI - HAVALIMANI_YENIKAPI,YENIKAPI - HAVALIMANI_ZEYTINBURNU,YENIKAPI-ADALAR_YENI KABATAS,YENIKOY-BEYKOZ_YENIKOY,day_of_week,month,hour,is_weekend
0,2023-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.081545,0.000000,0.158055,...,0.032043,0.005076,0.171894,0.042054,0.000000,0.000000,6,1,0,1
1,2023-01-01 01:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.115880,0.000000,0.000000,...,0.011126,0.002307,0.236534,0.035850,0.000000,0.000000,6,1,1,1
2,2023-01-01 02:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.012876,0.000000,0.051672,...,0.016021,0.000461,0.162742,0.035505,0.000000,0.000000,6,1,2,1
3,2023-01-01 03:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.005340,0.000461,0.108590,0.071699,0.000000,0.000000,6,1,3,1
4,2023-01-01 04:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.006676,0.001384,0.051578,0.015857,0.000000,0.000000,6,1,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8734,2023-12-31 19:00:00,0.090103,0.032389,0.064713,0.052676,0.045000,0.021073,0.017167,0.080824,0.039514,...,0.258567,0.072450,0.368958,0.258876,0.333333,0.179245,6,12,19,1
8735,2023-12-31 20:00:00,0.036928,0.000000,0.038665,0.026756,0.023125,0.000000,0.000000,0.044374,0.024316,...,0.169559,0.062298,0.331681,0.265081,0.000000,0.188679,6,12,20,1
8736,2023-12-31 21:00:00,0.028065,0.020243,0.031339,0.016722,0.000000,0.017241,0.008584,0.039620,0.027356,...,0.123275,0.048916,0.231862,0.146846,0.000000,0.150943,6,12,21,1
8737,2023-12-31 22:00:00,0.026588,0.000000,0.001221,0.000000,0.000000,0.000000,0.047210,0.017433,0.030395,...,0.109924,0.046147,0.215559,0.098931,0.000000,0.117925,6,12,22,1


In [13]:
list(X.columns)


['month', 'day_of_week', 'hour', 'is_weekend']

In [ ]:
import pandas as pd
import joblib
from datetime import timedelta

print(" Modeller hafızaya yükleniyor...")
all_models = joblib.load('all_istanbul_models.pkl')

def get_smart_route_prediction(route_stations, departure_time_str):
    """
    route_stations: Gidilecek durakların listesi (Sırayla)
    departure_time_str: Yola çıkış saati
    """
    current_time = pd.to_datetime(departure_time_str)
    total_duration_minutes = 0
    route_details = []
    
    base_time_per_stop = 2 
    
    print(f"\n🗺️ ROTA ANALİZİ: {departure_time_str} çıkışlı")
    print("-" * 50)
    
    for i, station in enumerate(route_stations):
        input_data = pd.DataFrame([{
            'month': current_time.month,
            'day_of_week': current_time.dayofweek,
            'hour': current_time.hour,
            'is_weekend': 1 if current_time.dayofweek >= 5 else 0
        }])
        
        if station in all_models:
            density = all_models[station].predict(input_data)[0]
        else:
            density = 0.5 
            
        # 3. Dinamik Süre Hesabı 
        # Formül: Normal Süre * (1 + Yoğunluk)
        # Eğer durak boşsa (0.1) -> 2 dk * 1.1 = 2.2 dk
        # Eğer durak patlıyorsa (0.9) -> 2 dk * 1.9 = 3.8 dk (Neredeyse 2 katı)
        
        real_duration = base_time_per_stop * (1 + density)
        total_duration_minutes += real_duration
        
        # Bir sonraki durağa varış zamanını güncelle
        current_time += timedelta(minutes=real_duration)
        
        status = "🟢" if density < 0.4 else "🟡" if density < 0.7 else "🔴"
        
        route_details.append({
            "station": station,
            "estimated_arrival": current_time.strftime("%H:%M"),
            "density_level": f"%{int(density*100)}",
            "status": status
        })
        
        print(f"{status} {station} -> Yoğunluk: %{int(density*100)} | Geçiş Süresi: {real_duration:.1f} dk")

    print("-" * 50)
    print(f"🏁 Toplam Tahmini Süre: {int(total_duration_minutes)} dakika")
    return route_details
my_route = [
    'YENIKAPI - HAVALIMANI_ZEYTINBURNU', 
    'YENIKAPI - HAVALIMANI_ULUBATLI',
    'YENIKAPI - HAVALIMANI_EMNIYET-FATIH'
]

get_smart_route_prediction(my_route, "2025-12-28 03:00:00")

get_smart_route_prediction(my_route, "2025-12-29 08:00:00")

 Modeller hafızaya yükleniyor...

🗺️ ROTA ANALİZİ: 2025-12-28 03:00:00 çıkışlı
--------------------------------------------------
🟢 YENIKAPI - HAVALIMANI_ZEYTINBURNU -> Yoğunluk: %0 | Geçiş Süresi: 2.0 dk
🟢 YENIKAPI - HAVALIMANI_ULUBATLI -> Yoğunluk: %0 | Geçiş Süresi: 2.0 dk
🟡 YENIKAPI - HAVALIMANI_EMNIYET-FATIH -> Yoğunluk: %50 | Geçiş Süresi: 3.0 dk
--------------------------------------------------
🏁 Toplam Tahmini Süre: 7 dakika

🗺️ ROTA ANALİZİ: 2025-12-29 08:00:00 çıkışlı
--------------------------------------------------
🟡 YENIKAPI - HAVALIMANI_ZEYTINBURNU -> Yoğunluk: %63 | Geçiş Süresi: 3.3 dk
🔴 YENIKAPI - HAVALIMANI_ULUBATLI -> Yoğunluk: %78 | Geçiş Süresi: 3.6 dk
🟡 YENIKAPI - HAVALIMANI_EMNIYET-FATIH -> Yoğunluk: %50 | Geçiş Süresi: 3.0 dk
--------------------------------------------------
🏁 Toplam Tahmini Süre: 9 dakika


[{'station': 'YENIKAPI - HAVALIMANI_ZEYTINBURNU',
  'estimated_arrival': '08:03',
  'density_level': '%63',
  'status': '🟡'},
 {'station': 'YENIKAPI - HAVALIMANI_ULUBATLI',
  'estimated_arrival': '08:06',
  'density_level': '%78',
  'status': '🔴'},
 {'station': 'YENIKAPI - HAVALIMANI_EMNIYET-FATIH',
  'estimated_arrival': '08:09',
  'density_level': '%50',
  'status': '🟡'}]